In [7]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# 1. Nosso DataFrame original do Pandas (criado anteriormente)
dados_reais = {
    "Nome": ["Hospital das Clínicas (HC)", "Colégio Bandeirantes", "Parque Ibirapuera"],
    "Tipo": ["Hospital", "Escola", "Parque"],
    "Latitude": [-23.5573, -23.5721, -23.5874],
    "Longitude": [-46.6661, -46.6508, -46.6576]
}
df_reais = pd.DataFrame(dados_reais)

# 2. TRANSFORMANDO EM GEOPANDAS:
# Usamos gpd.points_from_xy para converter as colunas numéricas em objetos de Ponto Geográfico
gdf_reais = gpd.GeoDataFrame(
    df_reais, 
    geometry=gpd.points_from_xy(df_reais.Longitude, df_reais.Latitude),
    crs="EPSG:4326"  # Define o sistema de coordenadas padrão mundial (WGS84)
)

# 3. Exibindo o novo resultado
print(gdf_reais)

                         Nome      Tipo  Latitude  Longitude  \
0  Hospital das Clínicas (HC)  Hospital  -23.5573   -46.6661   
1        Colégio Bandeirantes    Escola  -23.5721   -46.6508   
2           Parque Ibirapuera    Parque  -23.5874   -46.6576   

                    geometry  
0  POINT (-46.6661 -23.5573)  
1  POINT (-46.6508 -23.5721)  
2  POINT (-46.6576 -23.5874)  


In [8]:
# Mudar o sistema de coordenadas para metros (EPSG:31983 - ideal para o estado de SP)
gdf_metros = gdf_reais.to_crs(epsg=31983)

# Pegar a geometria da Escola (índice 1) e do Parque (índice 2)
escola_geom = gdf_metros.loc[1, 'geometry']
parque_geom = gdf_metros.loc[2, 'geometry']

# Calcular a distância em linha reta
distancia = escola_geom.distance(parque_geom)

print(f"A distância em linha reta da Escola até o Parque é de: {distancia:.2f} metros.")

A distância em linha reta da Escola até o Parque é de: 1831.08 metros.


In [3]:
import folium
import pandas as pd

# 1. Estrutura de dados com localizações reais em São Paulo
dados_reais = {
    "Nome": [
        "Hospital das Clínicas (HC)", 
        "Colégio Bandeirantes", 
        "Parque Ibirapuera"
    ],
    "Tipo": ["Hospital", "Escola", "Parque"],
    
    # Coordenadas geográficas reais (Latitude e Longitude)
    "Latitude": [-23.5573, -23.5721, -23.5874],
    "Longitude": [-46.6661, -46.6508, -46.6576],
    
    # Customização visual para o Folium
    "Cor_Marcador": ["red", "blue", "green"],
    "Icone": ["plus-sign", "education", "tree"]
}

# Convertendo para DataFrame do Pandas
df_reais = pd.DataFrame(dados_reais)
df_reais.head()

,Nome,Tipo,Latitude,Longitude,Cor_Marcador,Icone
0,Hospital das Clínicas (HC),Hospital,-23.5573,-46.6661,red,plus-sign
1,Colégio Bandeirantes,Escola,-23.5721,-46.6508,blue,education
2,Parque Ibirapuera,Parque,-23.5874,-46.6576,green,tree


In [5]:
# 2. Criando o mapa base centralizado na média das coordenadas reais
mapa_real = folium.Map(
    location=[df_reais["Latitude"].mean(), df_reais["Longitude"].mean()],
    zoom_start=13
)

# 3. Adicionando os marcadores reais com ícones personalizados
for index, linha in df_reais.iterrows():
    folium.Marker(
        location=[linha["Latitude"], linha["Longitude"]],
        popup=f"<b>{linha['Nome']}</b><br>Tipo: {linha['Tipo']}",
        tooltip=linha["Nome"],
        icon=folium.Icon(color=linha["Cor_Marcador"], icon=linha["Icone"], prefix="glyphicon")
    ).add_to(mapa_real)

# 4. Renderizar o mapa
mapa_real

In [9]:
import folium
import pandas as pd

# 1. Criando a tabela de dados usando o Pandas
dados = {
    'id': [1, 2, 3],
    'nome': ['Parque do Povo', 'Catedral da Sé', 'Hoapital da Sé'],
    'tipo': ['Parque', 'Igreja', 'Hospital'],
    'latitude': [-23.5505, -23.5512, -23.5528],
    'longitude': [-46.6333, -46.6345, -46.6310],
    'icone': ['tree', 'school', 'hospital'],  
    'cor': ['green', 'blue', 'red']           
}

df = pd.DataFrame(dados)

# 2. Criando o mapa base centralizado na média das coordenadas
# (Neste exemplo, as coordenadas são no centro de São Paulo)
mapa = folium.Map(location=[-23.5515, -46.6330], zoom_start=17)

# 3. Percorrendo a tabela (DataFrame) para adicionar os marcadores
for index, linha in df.iterrows():
    folium.Marker(
        location=[linha['latitude'], linha['longitude']],
        popup=f"<strong>{linha['nome']}</strong><br>Tipo: {linha['tipo']}",
        tooltip=linha['nome'],
        icon=folium.Icon(color=linha['cor'], icon=linha['icone'], prefix='fa')
    ).add_to(mapa)

mapa

In [7]:
import folium
import geopandas as gpd
from shapely import wkt

dados = {
    'id': [1, 2, 3],
    'nome': ['Parque do Povo', 'Catedral da Sé', 'Hoapital da Sé'],
    'tipo': ['Parque', 'Igreja', 'Hospital'],
    'geom': [
        'POINT(-46.6333 -23.5505)', 
        'POINT(-46.6345 -23.5512)', 
        'POLYGON((-46.6320 -23.5520, -46.6300 -23.5520, -46.6300 -23.5540, -46.6320 -23.5540, -46.6320 -23.5520))'
    ]
}

# 2. Convertemos o texto da coluna 'geom' para o formato espacial do GeoPandas
df = gpd.pd.DataFrame(dados)
df['geometry'] = df['geom'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

# 3. Criamos o mapa base
mapa = folium.Map(location=[-23.5520, -46.6325], zoom_start=16)

folium.GeoJson(
    gdf,
    name="Camada Geográfica",
    # Adicionamos estilo para que a área do parque fique verde e sem marcadores de balão genéricos
    style_function=lambda feature: {
        'fillColor': 'green' if feature['properties']['tipo'] == 'Parque' else 'blue',
        'color': 'darkgreen' if feature['properties']['tipo'] == 'Parque' else 'blue',
        'weight': 2,
        'fillOpacity': 0.4,
    },
    tooltip=folium.GeoJsonTooltip(fields=['nome', 'tipo'], aliases=['Nome:', 'Tipo:'])
).add_to(mapa)



# Mostrar o mapa direto no Jupyter
mapa

In [18]:
import folium
import geopandas as gpd
from shapely import wkt

p1 = " -46.634217 -23.550570" 
p2 = " -46.63360053833922 -23.54933205511358"
p3 = "-46.633228805146835 -23.5494607148146 "
p4 = " -46.63337294667658 -23.549898852638414"
p5 = "-46.63271292703931 -23.55011096547952 "
p6 = " -46.632030149541784 -23.54987102839663"
p7 = "-46.63195428023543 -23.55033002945189 "
p8 = " -46.63263388286607 -23.55169689682459"
p9 = " -46.63397363786296 -23.551163462059808 "
p10 = " -46.633784179262726 -23.550762348392283"

geometria_parque_real = f"POLYGON(({p1}, {p2}, {p3}, {p4}, {p5}, {p6},{p7}, {p8}, {p9}, {p10}, {p1}))"
dados = {
    'id': [1, 2, 3],
    'nome': ['Hospital Central', 'Escola Dom Pedro', 'Parque do Povo'],
    'tipo': ['Hospital', 'Escola', 'Parque'],
    'geom': [
        'POINT(-46.6333 -23.5505)',
        'POINT(-46.6345 -23.5512)',
        geometria_parque_real 
    ]
}

df = gpd.pd.DataFrame(dados)
df['geometry'] = df['geom'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")

mapa = folium.Map(location=[-23.5525, -46.6320], zoom_start=16)
folium.GeoJson(
    gdf,
    name="Camada Geográfica",
    style_function=lambda feature: {
        'fillColor': 'green' ,
        'color': 'darkgreen',
        'weight': 1,
        'fillOpacity': 0.4,
    },
    tooltip=folium.GeoJsonTooltip(fields=['nome', 'tipo'], aliases=['Nome:', 'Tipo:'])
).add_to(mapa)

mapa